# Character RNN for Generating Text

In [136]:
pip install torchtext==0.17.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 5.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 94.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 14.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [139]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [140]:
import torch
import torch.nn.functional as F
import time
import random
import unidecode
import string
import random
import re


torch.backends.cudnn.deterministic = True

## General Settings

In [151]:
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cpu")

TEXT_PORTION_SIZE = 200

NUM_ITER = 1000
LEARNING_RATE = 0.005
EMBEDDING_DIM = 100
HIDDEN_DIM = 100
NUM_HIDDEN = 1

## Dataset

Download *[A Tale of Two Cities](http://www.gutenberg.org/files/98/98-0.txt)* by Charles Dickens from the Gutenberg Project:

In [152]:
!wget http://www.gutenberg.org/files/98/98-0.txt

--2025-09-27 08:18:08--  http://www.gutenberg.org/files/98/98-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.gutenberg.org/files/98/98-0.txt [following]
--2025-09-27 08:18:08--  https://www.gutenberg.org/files/98/98-0.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 807231 (788K) [text/plain]
Saving to: ‘98-0.txt.2’

98-0.txt.2          100%[===================>] 788.31K  2.59MB/s    in 0.3s    

2025-09-27 08:18:09 (2.59 MB/s) - ‘98-0.txt.2’ saved [807231/807231]



Convert all characters into ASCII characters provided by `string.printable`:

In [153]:
string.printable

'0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~ \t\n\r\x0b\x0c'

In [154]:
with open('./98-0.txt', 'r') as f:
    textfile = f.read()

# convert special characters
textfile = unidecode.unidecode(textfile)

# strip extra whitespaces
textfile = re.sub(' +',' ', textfile)

TEXT_LENGTH = len(textfile)

print(f'Number of characters in text: {TEXT_LENGTH}')

Number of characters in text: 776562


In [155]:
random.seed(RANDOM_SEED)

def random_portion(textfile):
    start_index = random.randint(0, TEXT_LENGTH - TEXT_PORTION_SIZE)
    end_index = start_index + TEXT_PORTION_SIZE + 1
    return textfile[start_index:end_index]

print(random_portion(textfile))

 dancing, a dozen
together. When the wine was gone, and the places where it had been
most abundant were raked into a gridiron-pattern by fingers, these
demonstrations ceased, as suddenly as they had br


In [156]:
def char_to_tensor(text):
    lst = [string.printable.index(c) for c in text]
    tensor = torch.tensor(lst).long()
    return tensor

print(char_to_tensor('abcDEF'))

tensor([10, 11, 12, 39, 40, 41])


In [157]:
def draw_random_sample(textfile):    
    text_long = char_to_tensor(random_portion(textfile))
    inputs = text_long[:-1]
    targets = text_long[1:]
    return inputs, targets

In [158]:
draw_random_sample(textfile)

(tensor([11, 10, 23, 20, 73, 94, 10, 23, 13, 94, 27, 14, 31, 14, 10, 21, 94, 29,
         24, 94, 48, 27, 75, 94, 47, 24, 27, 27, 34, 94, 29, 17, 14, 94, 11, 27,
         18, 16, 17, 29, 23, 14, 28, 28, 96, 24, 15, 94, 29, 17, 14, 94, 54, 24,
         17, 24, 94, 17, 24, 27, 18, 35, 24, 23, 75, 94, 54, 24, 73, 94, 17, 14,
         94, 25, 30, 28, 17, 14, 13, 94, 24, 25, 14, 23, 94, 29, 17, 14, 94, 13,
         24, 24, 27, 94, 32, 18, 29, 17, 94, 29, 17, 14, 94, 32, 14, 10, 20, 94,
         27, 10, 29, 29, 21, 14, 96, 18, 23, 94, 18, 29, 28, 94, 29, 17, 27, 24,
         10, 29, 73, 94, 28, 29, 30, 22, 11, 21, 14, 13, 94, 13, 24, 32, 23, 94,
         29, 17, 14, 94, 29, 32, 24, 94, 28, 29, 14, 25, 28, 73, 94, 16, 24, 29,
         94, 25, 10, 28, 29, 94, 29, 17, 14, 94, 29, 32, 24, 94, 10, 23, 12, 18,
         14, 23, 29, 96, 12, 10, 28, 17, 18, 14, 27, 28, 73, 94, 10, 23, 13, 94,
         28, 17]),
 tensor([10, 23, 20, 73, 94, 10, 23, 13, 94, 27, 14, 31, 14, 10, 21, 94, 29, 24,
         

## Model

In [159]:
class RNN(torch.nn.Module):
    def __init__(self, input_size, embed_size,
                 hidden_size, output_size, num_layers):
        super(RNN, self).__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        self.embed = torch.nn.Embedding(input_size, hidden_size)
        self.gru = torch.nn.GRU(input_size=embed_size,
                                hidden_size=hidden_size,
                                num_layers=num_layers)
        self.fc = torch.nn.Linear(hidden_size, output_size)
        self.init_hidden = torch.nn.Parameter(torch.zeros(
                                              num_layers, 1, hidden_size))
    
    def forward(self, features, hidden):
        embedded = self.embed(features.view(1, -1))
        output, hidden = self.gru(embedded.view(1, 1, -1), hidden)
        output = self.fc(output.view(1, -1))
        return output, hidden
      
    def init_zero_state(self):
        init_hidden = torch.zeros(self.num_layers, 1, self.hidden_size).to(DEVICE)
        return init_hidden

In [160]:
torch.manual_seed(RANDOM_SEED)
model = RNN(len(string.printable), EMBEDDING_DIM, HIDDEN_DIM, len(string.printable), NUM_HIDDEN)
model = model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Training

In [162]:
def evaluate(model, prime_str='A', predict_len=100, temperature=0.8):

    hidden = model.init_zero_state()
    prime_input = char_to_tensor(prime_str)
    predicted = prime_str

    for p in range(len(prime_str) - 1):
        _, hidden = model(prime_input[p].to(DEVICE), hidden.to(DEVICE))
    inp = prime_input[-1]
    
    for p in range(predict_len):
        output, hidden = model(inp.to(DEVICE), hidden.to(DEVICE))
        
        # Sample from the network as a multinomial distribution
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        # Add predicted character to string and use as next input
        predicted_char = string.printable[top_i]
        predicted += predicted_char
        inp = char_to_tensor(predicted_char)

    return predicted

In [163]:
start_time = time.time()
for iteration in range(NUM_ITER):
    hidden = model.init_zero_state()
    optimizer.zero_grad()
    
    loss = 0.
    inputs, targets = draw_random_sample(textfile)
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    for c in range(TEXT_PORTION_SIZE):
        outputs, hidden = model(inputs[c], hidden)
        loss += F.cross_entropy(outputs, targets[c].view(1))

    loss /= TEXT_PORTION_SIZE
    loss.backward()
    
    ### UPDATE MODEL PARAMETERS
    optimizer.step()

    ### LOGGING
    with torch.set_grad_enabled(False):
      if iteration % 1000 == 0:
          print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')
          print(f'Iteration {iteration} | Loss {loss.item():.2f}\n\n')
          print(evaluate(model, 'Th', 200), '\n')
          print(50*'=')

Time elapsed: 0.00 min
Iteration 0 | Loss 4.60


ThusoGf&ZW@IB	'fYyB;SYkU@\#j4brzo'R+oy3x5Xid>qMP):f*!=/^d@"7lvFQh>	Yn:Xi^F/c']a!Vlj2|buznK);m-XrAn%!`|CVWj	m;Qy7S cDZ8MHq+'C&m5_Th_aYb8.@\^gf+v[Sk#@eAz
a*Ez-;,YUgq_QI#fRIMC+=KD3v@al1u%42C'TRwqVE=x 

